# Securitisation Lexicons Creation
This notebook contains the pipeline I use for creating the intial 3 lexicons capturing to the 3 dimensions of Securitisation Theory (Buzan et al, 1998), namely Threat, Urgency and Justification of Extraordinary Power. This feeds off one of Securitisation Theory's core concepts, that security isn't just a state of being but a performative act.

I ground this pipeline in an academically validated approach where I begin with seed words extracted from illlustative examples from foundational securitisation literature and then expand with relevant well-established lexicons, and finally grow it further using SBERT to find similar words used in similar contexts across the manifesto corpus. The result at the end of this pipeline is a transparent and theoretically grounded set lexicons capturing securitisation theory in its 3 consistutent parts.

In [1]:
#!pip install pyspellchecker inflect wordfreq xlsxwriter spellchecker
import string
from spellchecker import SpellChecker
import pandas as pd
import sys
import inflect
import wordfreq

# 1. Seed Words
Securitisation Theory is heavily based on discursive patterns and language purpose, so the formation of seed terms for each dimension will be reliant on a contextual inference process (as with Baele & Rousseau, 2022) where I choose words chosen from key examples in securitisation literature.

One thing not accounted for is pronouns. These could maybe tell an interesting story, like the use of "we" to unify the speaker with the audience, and "they" to discriminate certain groups. There is room to explore this in future work.

### 1a. Initial Lists Creation from Literature
Firstly the initial terms and phrases from the literature are extracted and supplemented with my own understanding of Securitisation Theory. Importantly these include words and phrases commonly utilised in securitising stylistic devices like metaphors (e.g. "wave" and "a cancer") and historical references (e.g. "9/11", "Blitz") (Baele & Rousseau, 2022). The former can give a simplified understanding which warrants a specific course of action and triggers certain emotions, and the latter projects a simplified version of a past event onto the present to unify the speaker with the audience and provide a course of actions based on past learning.

The following terms have been organised alphabetically for easy lookup if used in later work.

In [2]:
# Terms taken from: Buzan et al. (1998), (Otukoya (2024), Baele & Rousseau (2022)
threat_seed_lexicon_words= {
"7/7", "9/11", "abm", "aegis", "afghanistan", "aggression", "airpower", "alqaeda", "antipersonnel", "arab", "armament", "armaments",
"armies", "arming", "armory", "armour", "armoured", "armoury", "arms", "army", "assassin", "assassinate",
"assassinated", "assassinating", "assassination", "assassinations", "assassins", "assault", "atomic", "attack",
"attacked", "attacker", "attackers", "attacking", "attacks", "authority", "ballistic", "barbarian",
"barbarians", "battle", "biological", "blitz", "blockade", "blockaded", "blockades", "blockading", "blocked",
"blocks", "bombard", "bombarded", "bombarding", "bombardment", "bombardments", "bombards", "bombed", "bomber",
"bombers", "bombing", "bombings", "bombs", "bombshell", "bombshells", "brutal", "bullet", "capitulate",
"capitulated", "capitulating", "capitulation", "casualties", "casualty", "catastrophe", "catastrophes",
"catastrophic", "catastrophically", "chechnya", "chemical", "cluster", "coerce", "coerced", "coercing",
"coercion", "coercive", "colonial", "combat", "combatant", "combatants", "combated", "combating", "combats",
"combatting", "conflict", "conflicts", "counterfeit", "counterfeits", "coup", "crime", "crimea", "crimean",
"crimes", "criminal", "criminalisation", "criminalise", "criminalised", "criminalization", "criminalize",
"criminalized", "criminalizes", "criminalizing", "criminally", "criminals", "crisis", "cybersphere", "cyberwar",
"daesh", "death", "deaths", "demise", "demises", "destabilise", "destabilising", "destabilization",
"destabilize", "destabilized", "destabilizing", "destructed", "destructible", "destruction", "destructions",
"destructive", "destructively", "destructor", "deteriorate", "deteriorated", "deteriorates", "deteriorating",
"deterioration", "dictator", "disaster", "disease", "disintegrate", "disintegrated", "disintegrates",
"disintegrating", "disintegration", "disobedience", "disobedient", "division", "dominance", "dominant",
"dominantly", "dominate", "dominates", "dominating", "domination", "dominator", "dominators", "drowning",
"drownings", "drugs", "enemies", "enemy", "espionage", "espionages", "evil", "existential", "explosion",
"explosions", "explosive", "explosively", "explosives", "extinction", "extinctions", "extremism", "extremisms",
"extremist", "extremists", "faction", "fatalities", "fatality", "fighter", "fighters", "fighting", "fights",
"firearms", "fln", "forceful", "forcefully", "forces", "fragile", "fundamentalism", "fundamentalist",
"fundamentalists", "gang", "gaza", "genocide", "grave", "guerrilla", "guevara", "gulf", "hamas", "hazard",
"hazardous", "hazards", "hegemonic", "hegemony", "hezbollah", "hiroshima", "hostile", "hostiles", "hussein",
"illegal", "illegality", "illegally", "illegals", "illicit", "infiltrate", "infiltrated", "infiltrates",
"infiltrating", "infiltration", "infiltrations", "infiltrator", "infiltrators", "inhuman", "insecure",
"insecurities", "insecurity", "insurgencies", "insurgency", "insurgent", "insurgents", "interpol", "invaded",
"invader", "invaders", "invades", "invading", "invasion", "invasions", "ira", "iran", "iraq", "islamic",
"islamist", "israel", "israeli", "israelis", "israelite", "israelites", "israels", "killed", "killer",
"killers", "killing", "killings", "kills", "killswitch", "killzone", "kurd", "kurdish", "kurdistan", "kurds",
"landmine", "lawbreakers", "lawless", "lawlessness", "lebanon", "libya", "menace", "menaces", "militaries",
"militarisation", "militarised", "militarism", "militarist", "militaristic", "militarization", "militarized",
"military", "missile", "monster", "monsters", "munitions", "muslims", "nagasaki", "narcotics",
"nationalisation", "nationalist", "nationalistic", "nationalists", "nato", "nazi", "netwar", "nuclear", "osce",
"pakistan", "palestine", "partisan", "peril", "perils", "proliferating", "proliferation", "racism", "racist",
"racists", "radicalisation", "radicalise", "radicalised", "radicalising", "radicalism", "radicalization",
"radicalize", "radicalized", "radicalizing", "radicals", "repressive", "rival", "rivals", "rogue", "ruthless",
"rwanda", "rwandan", "salw", "salwar", "sarin", "seapower", "securities", "separatism", "separatist",
"separatists", "shock", "shortage", "somalia", "strike", "suicide", "superpower", "surveillance", "survival",
"syria", "target", "terminated", "termination", "terror", "terrorise", "terrorised", "terrorising", "terrorism",
"terrorist", "terroristic", "terrorists", "terrorize", "terrorized", "terrorizes", "terrorizing", "terrors",
"threat", "threaten", "threatened", "threatening", "threateningly", "threatens", "threats", "trafficked",
"trafficker", "traffickers", "trafficking", "traffics", "uprising", "vicious", "vietnam", "violence", "violent",
"violently", "virus", "viruses", "vulnerabilities", "vulnerability", "vulnerable", "vulnerables", "war",
"warfare", "warfield", "warfighter", "warfighting", "wargame", "wargames", "warhead", "warheads", "warmonger",
"warmongering", "warmongers", "warpath", "warplane", "warplanes", "wars", "wartime", "wave", "waves",
"weaponise", "weaponised", "weaponize", "weaponized", "weaponry", "weapons", "wmd", "wound", "wounded",
"wounding", "wounds"}

threat_seed_lexicon_terms= {"a cancer", "arm industry", "at war", "at wars", "being radicalised", "cogs of war", "cogs of wars", "day slavery", "engine of war",
"existential threat", "existential threats", "fight back", "fuel conflicts", "illegal immigrant",
"illegal immigrants", "illegal migrant", "illegal migrants", "invisible enemies", "invisible enemy",
"new verdun", "new verduns", "nine eleven", "nine-eleven", "our threat", "pearl harbor", "prisoners of",
"sea power", "sea powers", "security priorities", "serious threat", "serious threats", "seven seven",
"seven-seven", "the troubles", "toxic legacies", "under attack", "under attacks", "war engine", "war engines",
"war machine", "war machines", "war on", "wars on", "world war", "world wars"}

In [3]:
# Terms taken from: Buzan et al. (1998), Otukoya (2024), Ahmad (2025), Baele & Rousseau (2022)
urgency_seed_lexicon_words= {"armageddon", "collapse", "collapsed", "collapses", "collapsing", "crises", "crisis", "critical", "demand", "emergency", "ensure",
"escalate", "escalating", "escalation", "immediate", "immediately", "imminent", "imminently", "inevitabilities",
"inevitability", "inevitable", "irreversible", "mounting", "overdue", "overrun", "paramount", "react",
"reaction", "reactions", "reacts", "scarce", "scarcely", "scarcity", "spiralling", "swift", "tackle",
"unparalleled", "unprecedented", "urgent", "urgently", "worsening"}

urgency_seed_lexicon_terms= {"act now", "any longer", "before the election", "breaking point", "can't delay", "can't wait", "cannot afford", "cannot delay",
"cannot wait", "decisive action", "decisive actions", "end degrading", "from day one", "growing crisis",
"immediate action", "immediate actions", "last chance", "life threatening", "life-threatening", "make sure",
"move quickly", "move rapidly", "move swiftly", "must not delay", "mustn't delay", "national crisis",
"now or never", "on day one", "point of no return", "push to", "quickly move", "quickly moves", "rapidly move",
"rapidly moves", "strive for", "supreme priorities", "supreme priority", "swiftly move", "swiftly moves",
"threatening crisis", "ticking clock", "tipping point", "too late", "urgent priorities", "urgent priority",
"wakeup call", "without delay"}

In [4]:
# Terms taken from: Buzan et al. (1998), Otukoya (2024), State of exception (2003), Flohr (2025)
justification_of_extr_power_seed_lexicon_words= {"abate", "abatement", "abolish", "abolition", "anarchy", "armistice", "ceasefire", "compelling", "compels", "compromise", "concession",
"conciliation", "conciliatory", "containment", "controlling", "controls", "cooperate", "cooperated",
"cooperates", "cooperating", "cooperation", "counteracts", "counterattack", "counterattacks",
"counterinsurgency", "counterinsurgent", "counterterrorism", "countervail", "countervailing", "crackdown",
"crackdowns", "curb", "curfew", "curfews", "decree", "decrees", "defend", "demand", "deport", "deported",
"deter", "deteriorating", "deterred", "deterrence", "detterence", "detterent", "detterents", "disarmament",
"disarmed", "disarming", "disarms", "détente", "embargo", "enforce", "enforcement", "expedite", "forbid",
"forbidden", "forbidding", "forbids", "forced", "halt", "halting", "homeland", "hostage", "interdependence",
"interdependent", "interdiction", "intervene", "intervention", "introduce", "lawfully", "liberate", "liberated",
"liberates", "liberating", "liberation", "liberator", "liberators", "lockdown", "lockdowns", "mobilise",
"mobilises", "mobilize", "mobilizes", "monitor", "nonproliferation", "overdue", "overhaul", "oversee", "powers",
"preempt", "preempted", "preemption", "preemptive", "preemptively", "prerogative", "prerogatives", "prevent",
"preventable", "preventative", "preventing", "prevention", "preventive", "protect", "protection", "protections",
"protective", "protectively", "protector", "protectorate", "protectorates", "protectors", "protects",
"punitive", "remove", "rescue", "resist", "resistance", "resistances", "resisted", "resisting", "retaliate",
"retaliation", "revenge", "sanction", "sanctioned", "sanctioning", "sanctions", "securities", "security",
"securitys", "siege", "sieges", "sovereign", "sovereigns", "sovereignty", "stabilise", "stabilised",
"stabilising", "surveil", "surveillance", "surveillances", "surveils", "suspension", "suspensions", "tackle",
"tactical", "tactically", "tactician", "tactics", "toughen", "toughened", "toughening", "uncooperating",
"uncooperative", "worsening"}

justification_of_extr_power_seed_lexicon_terms= {"age limits", "age verification", "beyond politics", "border closure", "border closures", "break free", "breaking free", "can't accept",
"can't wait", "cannot wait", "close our border", "close our borders", "close the border", "close the borders",
"commander in chief", "commander-in-chief", "commanders-in-chief", "counter attack", "counter attacks",
"counter-attack", "counter-attacks", "crack down", "crack downs", "decisive action", "decisive actions",
"don't accept", "exceptional means", "exceptional measure", "exceptional measures", "extraordinary means",
"extraordinary measures", "fast track", "fight back", "maintain order", "martial law", "martial laws",
"not accept", "protect frontline", "protecting people", "restore order", "shut our border", "shut our borders",
"shut the border", "shut the borders", "solemn duties", "solemn duty", "state of emergencies",
"state of emergency", "stop brexit", "stop the boats", "stronger control", "stronger controls", "subject to",
"ticking clock", "tipping point", "tougher sentencing", "without delay", "won't accept"}

### 1b. Validation
All seed words above were manually entered by myself. Therefore, validation is needed to ensure they're in the correct format for subsequent pipeline steps.

In [5]:
# Ensure I spell all words correctly
spell_check_lexicon_words= SpellChecker()

# I 100% need British spellings too as these are UK manifestos
words_i_explicitly_correct= {'mobilise', 'mobilize', 'lockdown', 'counter-attack', 'lockdowns', 'espionages', 'invadings', 'vulnerables', 'new verduns',
                                 'counter-attacks', 'surveillances', 'mobilises', 'surveil', 'surveils', 'inevitabilities'}
spell_check_lexicon_words.known(list(words_i_explicitly_correct))

# i setup this helper-function to go through & systematically correct misspelt lexicon terms given one of the lexicons
def spell_check_and_correct_lexicon_words(lexicon_set, lexicon_name, explicitly_correct_words):
    lex_corrected_terms= set()
    misspelled_lex_terms= {}
    print(f"\n now spell checking for {lexicon_name}...")
    for term in lexicon_set:
        if term is None:
            continue # (if this crashed on None it has to be skipped)

        # I check if the entire term (single word or phrase) is explicitly accepted in the british spellings I note
        # If so I add it as is and skip further correction attempts.
        if term in explicitly_correct_words:
            lex_corrected_terms.add(term)
            continue

        # this section is for handling multi-word terms
        if ' ' in term:
            corrected_phrase_words= []
            original_words= term.split()
            lexicon_phrase_misspelled= False
            for word in original_words:
                # here I check if individual word in phrase is explicitly accepted for british spelling
                if word in explicitly_correct_words:
                    corrected_word= word
                else:
                    corrected_word= spell_check_lexicon_words.correction(word)
                    if corrected_word is None: # I have this fallback in the case of no correction needed
                        corrected_word= word
                        
                if corrected_word != word:
                    lexicon_phrase_misspelled= True
                corrected_phrase_words.append(corrected_word)
            corrected_term_str= ' '.join(corrected_phrase_words)

            if lexicon_phrase_misspelled:
                misspelled_lex_terms[term]= corrected_term_str
                lex_corrected_terms.add(corrected_term_str)
            else:
                lex_corrected_terms.add(term)
        # and now i do the same for single words
        else:
            corrected_word= spell_check_lexicon_words.correction(term)
            if corrected_word is None:
                corrected_word= term

            if corrected_word != term:
                misspelled_lex_terms[term]= corrected_word
                lex_corrected_terms.add(corrected_word)
            else:
                lex_corrected_terms.add(term)

    if misspelled_lex_terms:
        print(f"the misspelled words in {lexicon_name} are: {misspelled_lex_terms}")
        print(f"so the corrected ones in {lexicon_name} were: {lex_corrected_terms}")
    else:
        print(f"there's actually no misspelled words found in the {lexicon_name}.")
    return lex_corrected_terms

In [6]:
# this is a validation barrier I setup to ensure all seed terms are correctly formatted i.e. no capitilisation and punctuation, + correct spellings.
def validate_lexicon_entries(lexicon_set, lexicon_name, words_i_explicitly_correct):
  spell_check_and_correct_lexicon_words(lexicon_set, lexicon_name, words_i_explicitly_correct)
  for lex_entry in lexicon_set:
    lex_entry= lex_entry.lower()
    lex_entry= lex_entry.translate(str.maketrans('', '', string.punctuation))
  print(f"Great, the {lexicon_name} lexicon is now all validated.")

# now I just run that validation barrier on all the lexicon words & terms
validate_lexicon_entries(threat_seed_lexicon_words, "threat_seed_lexicon_words", words_i_explicitly_correct)
validate_lexicon_entries(threat_seed_lexicon_terms, "threat_seed_lexicon_terms", words_i_explicitly_correct)
validate_lexicon_entries(urgency_seed_lexicon_words, "urgency_seed_lexicon_words", words_i_explicitly_correct)
validate_lexicon_entries(urgency_seed_lexicon_terms, "urgency_seed_lexicon_terms", words_i_explicitly_correct)
validate_lexicon_entries(justification_of_extr_power_seed_lexicon_words, "justification_of_extr_power_seed_lexicon_words", words_i_explicitly_correct)
validate_lexicon_entries(justification_of_extr_power_seed_lexicon_terms, "justification_of_extr_power_seed_lexicon_terms", words_i_explicitly_correct)


 now spell checking for threat_seed_lexicon_words...
the misspelled words in threat_seed_lexicon_words are: {'armoured': 'armored', 'wargames': "wagram's", 'criminalisation': 'criminalization', 'fln': 'fun', 'nato': 'nate', 'seapower': 'empower', 'warfield': 'garfield', 'extremisms': 'extremists', 'armoury': 'armour', 'militarised': 'militarized', 'terrorising': 'terrorizing', 'terrorise': 'terrorist', 'israelites': 'israelite', 'militarisation': 'militarization', 'terrorised': 'terrorized', 'dominators': 'dominator', 'wmd': 'wed', 'weaponized': 'caponized', 'radicalised': 'radicalized', 'alqaeda': 'alfreda', 'netwar': 'near', 'criminalise': 'criminalist', 'destabilising': 'destabilizing', 'osce': 'once', 'salwar': 'solar', 'destabilise': 'destabilize', 'criminalised': 'criminalized', 'abm': 'am', 'kurds': 'turds', 'weaponise': 'weapons', 'sarin': 'satin', 'wargame': 'warfare', 'daesh': 'dash', 'radicalisation': 'radicalization', 'radicalise': 'radicalism', 'weaponize': 'caponize', 'n

the misspelled words in justification_of_extr_power_seed_lexicon_words are: {'stabilised': 'stabilized', 'stabilise': 'stabilize', 'detterents': 'detergents', 'detterence': 'deference', 'detterent': 'detergent', 'stabilising': 'stabilizing', 'uncooperating': 'cooperating', 'securitys': 'security'}
so the corrected ones in justification_of_extr_power_seed_lexicon_words were: {'resistances', 'surveillance', 'preventive', 'disarms', 'defend', 'enforce', 'protectorates', 'nonproliferation', 'countervail', 'toughened', 'conciliatory', 'stabilize', 'sanctioned', 'counterinsurgency', 'surveils', 'resisted', 'sovereignty', 'controlling', 'stabilizing', 'mobilise', 'abolition', 'détente', 'protectively', 'tactician', 'remove', 'prerogative', 'interdependence', 'lockdowns', 'protection', 'disarming', 'forbidding', 'deteriorating', 'abatement', 'curfews', 'detergent', 'embargo', 'cooperation', 'crackdowns', 'tactical', 'compromise', 'deterred', 'overhaul', 'detergents', 'deport', 'punitive', 'lib

### 1c. Pluralising Root Words
By nature, any securitised word in its singular term will often still be securitised in its plural form. I desgined the following code to automatically pluralise these words to safely expand each lexicon set.

In [7]:
# this expands to pluralised terms I might have missed above
lexicon_pluraliser= inflect.engine()

def get_plural_forms_for_the_lexicon(word_set):
    pluralised_set = set(word_set)
    for word in word_set:
        # this now handles my multi-word terms by pluralising the last word if it's a noun
        if ' ' in word:
            parts_of_it= word.split()
            last_word= parts_of_it[-1]
            # I only attempt to pluralise if the last word is likely a noun (just a simple heuristic, could make it more complicated in future versions though)
            # I could even do POS tagging in future but this should still be fine. One this this doesn't catch is "war on"->"wars on" but it does do "war ons"
            if last_word and not last_word.isdigit() and not last_word.isupper() and not last_word.isdigit(): # Avoid pluralising acronyms or numbers
                plural_last_word= lexicon_pluraliser.plural(last_word)
                if plural_last_word != last_word: # here I only add if a different plural form is generated
                    pluralised_set.add(' '.join(parts_of_it[:-1] + [plural_last_word]))
        else:
            # this is the simpler case where I pluralise single words
            if not word.isdigit():
              plural_word= lexicon_pluraliser.plural(word)
              if plural_word != word:
                  pluralised_set.add(plural_word)
    return list(pluralised_set)

In [9]:
# I run that pluraliser to fetch & store all the plural words & terms for manual vetting
threat_seed_lexicon_words= get_plural_forms_for_the_lexicon(threat_seed_lexicon_words)
threat_seed_lexicon_terms= get_plural_forms_for_the_lexicon(threat_seed_lexicon_terms)
urgency_seed_lexicon_words= get_plural_forms_for_the_lexicon(urgency_seed_lexicon_words)
urgency_seed_lexicon_terms= get_plural_forms_for_the_lexicon(urgency_seed_lexicon_terms)
justification_of_extr_power_seed_lexicon_words= get_plural_forms_for_the_lexicon(justification_of_extr_power_seed_lexicon_words)
justification_of_extr_power_seed_lexicon_terms= get_plural_forms_for_the_lexicon(justification_of_extr_power_seed_lexicon_terms)

In [10]:
# 1st, all the sets are be displayed neatly
all_lexicons_flat_tiers= [
    {"Lexicon Name": "threat words", "Terms": ', '.join(sorted(list(threat_seed_lexicon_words)))},
    {"Lexicon Name": "threat terms", "Terms": ', '.join(sorted(list(threat_seed_lexicon_terms)))},
    {"Lexicon Name": "urgency words", "Terms": ', '.join(sorted(list(urgency_seed_lexicon_words)))},
    {"Lexicon Name": "urgency terms", "Terms": ', '.join(sorted(list(urgency_seed_lexicon_terms)))},
    {"Lexicon Name": "just. extr power words", "Terms": ', '.join(sorted(list(justification_of_extr_power_seed_lexicon_words)))},
    {"Lexicon Name": "just. extr power terms", "Terms": ', '.join(sorted(list(justification_of_extr_power_seed_lexicon_terms)))}
]
lexicon_df= pd.DataFrame(all_lexicons_flat_tiers)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
display(lexicon_df)

Lexicon Name  \
0            threat words   
1            threat terms   
2           urgency words   
3           urgency terms   
4  just. extr power words   
5  just. extr power terms   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [11]:
# some of the pluralised terms aren't suitable in context, so I manually remove them here to avoid clogging up the dictionaries with non-wordsl
remove_pluralised_threat_words = {"9/11S", "7/7S", "brutals", "catastrophics", "fragiles", "graves", "nine-elevens", "seven-sevens", "ruthlesses", "blitzzes", "rouges", "terminateds", "viciouses"}
remove_pluralised_threat_terms = {"new verdunes", "nine elevens", "pearl harbors", "seven sevens", "war ons", "wars ons", "a cancers", "engine of wars", "fight backs"}
threat_seed_lexicon_words = {word for word in threat_seed_lexicon_words if word not in remove_pluralised_threat_words}
threat_seed_lexicon_terms = {term for term in threat_seed_lexicon_terms if term not in remove_pluralised_threat_terms}

remove_pluralised_urgency_words = {"alarmings", "immediates", "paramounts", "unparalleleds", "unprecedenteds", "vitals", "crise", "absolutelies", "dangerouses",
                                   "urgentlies", "life-threatenings", "uncompromisinglies", "inevitables", "musts"}
remove_pluralised_urgency_terms = {"acts nows", "any longers", "can't delays", "can't waits", "cannot delays", "can't delays", "must act nows", "must not delays",
                                   "mustn't delays", "no longers", "point of no returns", "too lates", "cannot waits", "act nows", "life threatenings",
                                   "move swiftlies", "move rapidlies", "move quicklies"}
urgency_seed_lexicon_words = {word for word in urgency_seed_lexicon_words if word not in remove_pluralised_urgency_words}
urgency_seed_lexicon_terms = {term for term in urgency_seed_lexicon_terms if term not in remove_pluralised_urgency_terms}

remove_pluralised_justification_of_extr_power_words = {"compelleds"}
remove_pluralised_justification_of_extr_power_terms = {"above politic", "beyond politic", "breaking frees", "commander in chiefs", "exceptional mean", "extraordinary mean",
                                                       "maintain orders", "restore orders", "extraordinary measure", "fight backs"}
justification_of_extr_power_seed_lexicon_words = {term for term in justification_of_extr_power_seed_lexicon_words if term not in remove_pluralised_justification_of_extr_power_words}
justification_of_extr_power_seed_lexicon_terms = {term for term in justification_of_extr_power_seed_lexicon_terms if term not in remove_pluralised_justification_of_extr_power_terms}


# & quick sanity-check of the current state of the lexicons
print(f"Threat words: {threat_seed_lexicon_words}")
print(f"Threat terms: {threat_seed_lexicon_terms}")
print(f"Urgency words: {urgency_seed_lexicon_words}")
print(f"Urgency terms: {urgency_seed_lexicon_terms}")
print(f"Just. extr power words: {justification_of_extr_power_seed_lexicon_words}")
print(f"Just. extr power terms: {justification_of_extr_power_seed_lexicon_terms}")

Threat words: {'suicide', 'chemicals', 'killed', 'disintegration', 'natoes', 'gulf', 'warmongers', 'strike', 'kurdish', 'armoured', 'bombed', 'crimes', 'surveillance', 'warheads', 'wargames', 'bullet', 'brutal', 'nationalist', 'capitulating', 'casualties', 'guevaras', 'arming', 'drugs', 'combateds', 'destructible', 'faction', 'interpols', 'killeds', 'aegise', 'wars', 'traffickers', 'militarizations', 'blockade', 'dominances', 'gang', 'infiltrations', 'terrorizeds', 'biologicals', 'rivals', 'threatens', 'conflict', 'fln', 'attack', 'assault', 'illicits', 'infiltrateds', 'radicalizings', 'catastrophicallies', 'invaders', 'islamics', 'racist', 'islamists', 'violent', 'destruction', 'violently', 'militarized', 'invadeds', 'ruthless', 'trafficker', 'traffickeds', 'radicalize', 'coercing', 'nato', 'seapower', 'munitions', 'warplane', 'killzones', 'invader', 'weapon', 'landmines', 'lawlessnesses', 'target', 'destructeds', 'munition', 'hazardouses', 'existentials', 'guerrillas', 'narcotic', 'c

### 1d. Categorising Terms
Now these terms must be categorised based on contextual intensity.
1. Core signals unambiguously enact securitisation.
2. Secondary signals itensify/legitimise it
3. Contextual signals ("contingent lexicon" in literature) gain securitising power in context due to their specific socio-cultural resonance or historical weight.

These align with my research:

* Frame semantics (frame-evoking vs frame-supporting elements)
* Speech act theory (illocutionary force [intended action] vs perlocutionary reinforcement [effect from utterance])
* Discourse analysis: core claim vs justificatory scaffolding

I use these 3 signal types will to weight securitisation regression. They also allow me room for more in-depth analysis, but you can check out my paper for more details on that.

Grading between categories varies between dimensions as each one has distinctive features; threat establishes existential danger, urgency compresses time & pushes urgency, and justifying extyraordinary measures legitimises suspending normal politics.

**Threat**

| Signal Tier | Definition / Criteria | Validated Examples from Sources |
| :--- | :--- | :--- |
| **Core** | The vocabulary of survival: Words explicitly denoting the end of existence or presence of a fundamental danger. **Test:** Does this word frame something as a serious or existential threat without needing extra context? | "existential threat", "survival", "war" (generic lexicon), "peril", "extinction". |
| **Secondary** | The vocabularly of hostility: words that help to create a "Self-Other" dichotomy, assigning agency and malevolence to the threat to make it "actorised". **Test:** Does this word describe the threat as distinctively evil or aggressive? | "vicious", "brutal", "Invisible enemy", "infiltrating", "rogue", "barbarians", "counter-attack". |
| **Contextual** | The vocabularly of vulnerability. "Thick signifiers" that trigger specific cultural traumas or name the specific object at risk. **Test:** Is this word a historical event or a specific policy topic (e.g., health, migration) being framed as dangerous? | "pearl harbor", "the blitz", "9/11" (historical analogies); "virus", "migrant", "waves" (metaphor). |

**Urgency**

| Signal Tier | Definition / Criteria | Validated Examples from Sources |
| :--- | :--- | :--- |
| **Core** | The vocabulary of immediacy. Temporal markers that explicitly collapse the timeframe for decision-making, signalling that the opportunity to act is vanishing. **Test:** does this word assert that action must happen *now* or that time has run out? | "urgent", "immediate", "point of no return", "crisis", "supreme priority", "ticking clock". |
| **Secondary** | The vocabulary of priority and escalation. Words that elevate an issue above others, emphasising that something must be done soon to avoid a disaster, or that a situation is worsening. **Test:** does this word claim the issue is exceptional, deteriorating, or cannot be deferred? | "unprecedented", "vital", "paramount", "Inevitable". |
| **Contextual** | The vocabulary of commitment and insistence. Language expressing determination or resolve to act, which implies urgency through intensity of commitment without explicitly invoking time pressure or crisis. **Test:** does this word signal strong intent to act without specifying a timeframe? | "ensure", "strive to", "push for". |

**Justification of Extr. Power**

| Signal Tier | Definition / Criteria | Validated Examples from Sources |
| :--- | :--- | :--- |
| **Core** | The vocabulary of exception. Words that explicitly signal the suspension of the normal legal/political order. **Test:** Does this word legally or conceptually denote a break from the norm? | "state of emergency", "exceptional measures", "martial law", "breaking free", "suspension". |
| **Secondary** | Words that invoke institutional legitimacy, moral obligation, or the mandate to act decisively. **Test:** Does this word assert the power or responsibility to act beyond normal deliberation? | "commander in chief", "solemn duty", "order", "mobilise", "must", "compelled". |
| **Contextual** | The vocabulary of tools. Specific policy instruments that become "extraordinary" only when linked to the threat. In normal times, these might be administrative; in security times, they are survival tools. **Test:** Is this a specific policy tool (e.g., border closure, censorship)? | "lockdown", "surveillance", "quarantine", "border closure", "sanctions". |

In [12]:
# I now use dicts to create a mapping of words to signals
# Core signals should never really change, but the other dimensions will grow as pre-established lexicons and SBERT are utilised

threat_words_core_signals = {"survival", "survivals", "security", "securities", "threat", "peril", "perils", "extinction", "extinctions", "war", "wars",
                             "catastrophe", "catastrophic", "catastrophes", "death", "deaths", "fatalities", "fatality", "terminated", "termination",
                             "terminations", "wound", "wounds", "monster", "monsters", "threat", "threats", "invading", "invadings", "invasion", "invasions",
                             "attack", "attacks", "drowning", "drownings", "extremisms", "extremist", "extremists", "extremism"}
threat_terms_core_signals = {"existential threat", "existential threats", "serious threat", "serious threats", "war machine",
                             "war machines", "war engines", "war engine", "engine of war", "cogs of wars", "cogs of war" , "fight back"}

threat_words_secondary_signals = {"terrorist", "terrorists", "enemy", "enemies", "hostile", "hostiles", "vicious", "brutal",
                                  "infiltrating", "infiltration", "infiltrations", "infiltratings", "ruthless",  "counterfeit", "counterfeits", "rouge",
                                  "fragile", "vulnerable", "vulnerables", "vulnerability", "vulnerabilities", "demise", "demises", "erasure", "erasures",
                                  "espionage", "espionages", "barbarians", "barbarian", "illegal", "grave", "menace", "menaces",}
threat_terms_secondary_signals = {"rouge states", "rouge state", "under attack", "invisible enemy", "invisible enemies",}

threat_words_contextual_signals = {"9/11", "nine-eleven", "7/7", "seven-seven", "blitz", "blitzes", "illegals", "graves",
                                   "waves", "wave", "virus", "viruses", "rwanda", "rwandan"}
threat_terms_contextual_signals = {"nine eleven", "seven seven", "new verdun", "new verduns", "pearl harbor", "world war", "world wars", "at war", "war on",
                                   "wars on", "at wars", "under attacks", "a cancer", "sea powers", "sea power", "illegal immigrant", "illegal immigrants", "illegal migrant", "illegal migrants"}


urgency_words_core_signals = {"urgent", "urgents", "immediate", "crisis", "crises", "life-threatening", "unparalleled", "unprecedented"}
urgency_terms_core_signals = {"point of no return", "too late", "supreme priority", "supreme priorities", "urgent priority", "urgent priorities",
                              "life threatening", "act now", "must act now"}

urgency_words_secondary_signals = {"alarming", "vital", "paramount", "uncompromisingly", "urgently", "absolutely", "dangerous", "must", "deter", "deters",
                                   "prevent", "prevents", "react", "reacts", "reaction", "reactions"}
urgency_terms_secondary_signals = {"can't wait", "cannot wait", "can't delay", "cannot delay", "must not delay", "mustn't delay", "decisive action", "decisive actions",
                                   "immediate action",  "immediate actions",}

urgency_words_contextual_signals = {"inevitabilities", "inevitability", "inevitable", }
urgency_terms_contextual_signals = {"any longer", "no longer", "move quickly", "quickly move", "quickly moves", "swiftly move", "move swiftly", "swiftly moves",
                                    "rapidly move", "move rapidly", "rapidly moves", "before the election", "rapid production", "before the elections",
                                    "rapid productions", "ticking clock", "ticking clocks", }


justification_of_extr_power_words_core_signals = {"siege", "sieges", "curfew", "curfews", "counter-attack", "counter-attacks", "counterattacks", "counterattack"}
justification_of_extr_power_terms_core_signals = {"martial law", "martial laws", "state of emergency", "state of emergencies", "exceptional measures", "exceptional measure", "extraordinary measures",
                                                  "counter attack", "counter attacks", "exceptional means", "extraordinary means", "breaking free", "restore order", "fight back",}

justification_of_extr_power_words_secondary_signals = {"sovereign", "sovereigns", "prerogative", "prerogatives", "decree", "decrees", "protect", "protects",
                                                       "compelled", "commanders-in-chief", "commander-in-chief"}
justification_of_extr_power_terms_secondary_signals = {"commander in chief", "highest duty", "highest duties", "above politics", "beyond politics", "maintain order", "solemn duty", "solemn duties"}

justification_of_extr_power_words_contextual_signals = {"lockdown", "lockdowns", "mobilise", "mobilises", "mobilize", "mobilizes", "order", "orders",
                                                        "suspension", "suspensions", "surveillance", "surveillances", "sanction", "sanctions", "surveils", "surveil"}
justification_of_extr_power_terms_contextual_signals = {"close the border", "shut the border", "shut the borders", "border closure", "shut our borders", "border closures",
                                                        "close the borders", "shut our border", "close our border", "close our borders"}

# Now I filter out all words not present in the original set

# Threat words & term signal categories are filtered against threat_seed_lexicon_words
threat_words_core_signals= {word for word in threat_words_core_signals if word in threat_seed_lexicon_words}
threat_words_secondary_signals= {word for word in threat_words_secondary_signals if word in threat_seed_lexicon_words}
threat_words_contextual_signals= {word for word in threat_words_contextual_signals if word in threat_seed_lexicon_words}

threat_terms_core_signals= {term for term in threat_terms_core_signals if term in threat_seed_lexicon_terms}
threat_terms_secondary_signals= {term for term in threat_terms_secondary_signals if term in threat_seed_lexicon_terms}
threat_terms_contextual_signals= {term for term in threat_terms_contextual_signals if term in threat_seed_lexicon_terms}

# Urgency words & terms signal categories are filtered against urgency_seed_lexicon_words
urgency_words_core_signals= {word for word in urgency_words_core_signals if word in urgency_seed_lexicon_words}
urgency_words_secondary_signals= {word for word in urgency_words_secondary_signals if word in urgency_seed_lexicon_words}
urgency_words_contextual_signals= {word for word in urgency_words_contextual_signals if word in urgency_seed_lexicon_words}

urgency_terms_core_signals= {term for term in urgency_terms_core_signals if term in urgency_seed_lexicon_terms}
urgency_terms_secondary_signals= {term for term in urgency_terms_secondary_signals if term in urgency_seed_lexicon_terms}
urgency_terms_contextual_signals= {term for term in urgency_terms_contextual_signals if term in urgency_seed_lexicon_terms}

# Justification of Extr. Power words & terms signal categories are filtered against justification_of_extr_power_seed_lexicon_words
justification_of_extr_power_words_core_signals= {word for word in justification_of_extr_power_words_core_signals if word in justification_of_extr_power_seed_lexicon_words}
justification_of_extr_power_words_secondary_signals= {word for word in justification_of_extr_power_words_secondary_signals if word in justification_of_extr_power_seed_lexicon_words}
justification_of_extr_power_words_contextual_signals= {word for word in justification_of_extr_power_words_contextual_signals if word in justification_of_extr_power_seed_lexicon_words}

justification_of_extr_power_terms_core_signals= {term for term in justification_of_extr_power_terms_core_signals if term in justification_of_extr_power_seed_lexicon_terms}
justification_of_extr_power_terms_secondary_signals= {term for term in justification_of_extr_power_terms_secondary_signals if term in justification_of_extr_power_seed_lexicon_terms}
justification_of_extr_power_terms_contextual_signals= {term for term in justification_of_extr_power_terms_contextual_signals if term in justification_of_extr_power_seed_lexicon_terms}

In [13]:
# now using the categorisations above, I use this function to set them in their corresponding class
def categorise_and_report_lexicon(lexicon_set, core_signals, secondary_signals, contextual_signals, lexicon_name):
    categorised_lexicon= {}
    for term in lexicon_set:
        if term in core_signals:
            categorised_lexicon[term]= "core"
        elif term in secondary_signals:
            categorised_lexicon[term]= "secondary"
        elif term in contextual_signals:
            categorised_lexicon[term]= "contextual"
        else:
            categorised_lexicon[term]= "n/a"

    # I highlight any ones I could've missed here
    invalid_lex_terms= [term for term, signal_type in categorised_lexicon.items() if signal_type == 'n/a']
    if invalid_lex_terms:
        print(f"oops, in {lexicon_name} I forgot to assign a signal to these terms: {invalid_lex_terms}")
    else:
        print(f"All good in {lexicon_name}, I've assigned each term a signal category")
    return categorised_lexicon

# now I categorise threat words & terms
threat_seed_lexicon_words_signal_categories= categorise_and_report_lexicon(threat_seed_lexicon_words, threat_words_core_signals, threat_words_secondary_signals, threat_words_contextual_signals, "threat words")
threat_seed_lexicon_terms_signal_categories= categorise_and_report_lexicon(threat_seed_lexicon_terms, threat_terms_core_signals, threat_terms_secondary_signals, threat_terms_contextual_signals, "threat terms")

# ... & urgency ones
urgency_seed_lexicon_words_signal_categories= categorise_and_report_lexicon(urgency_seed_lexicon_words, urgency_words_core_signals, urgency_words_secondary_signals, urgency_words_contextual_signals, "urgency words")
urgency_seed_lexicon_terms_signal_categories= categorise_and_report_lexicon(urgency_seed_lexicon_terms, urgency_terms_core_signals, urgency_terms_secondary_signals, urgency_terms_contextual_signals, "urgency terms")

# & finally justification ones too
justification_of_extr_power_seed_lexicon_words_signal_categories= categorise_and_report_lexicon(justification_of_extr_power_seed_lexicon_words, justification_of_extr_power_words_core_signals, justification_of_extr_power_words_secondary_signals, justification_of_extr_power_words_contextual_signals, "justification words")
justification_of_extr_power_seed_lexicon_terms_signal_categories= categorise_and_report_lexicon(justification_of_extr_power_seed_lexicon_terms, justification_of_extr_power_terms_core_signals, justification_of_extr_power_terms_secondary_signals, justification_of_extr_power_terms_contextual_signals, "justification terms")


oops, in threat words I forgot to assign a signal to these terms: ['suicide', 'chemicals', 'killed', 'disintegration', 'natoes', 'gulf', 'warmongers', 'strike', 'kurdish', 'armoured', 'bombed', 'crimes', 'surveillance', 'warheads', 'wargames', 'bullet', 'nationalist', 'capitulating', 'casualties', 'guevaras', 'arming', 'drugs', 'combateds', 'destructible', 'faction', 'interpols', 'killeds', 'aegise', 'traffickers', 'militarizations', 'blockade', 'dominances', 'gang', 'terrorizeds', 'biologicals', 'rivals', 'threatens', 'conflict', 'fln', 'assault', 'illicits', 'infiltrateds', 'radicalizings', 'catastrophicallies', 'invaders', 'islamics', 'racist', 'islamists', 'violent', 'destruction', 'violently', 'militarized', 'invadeds', 'trafficker', 'traffickeds', 'radicalize', 'coercing', 'nato', 'seapower', 'munitions', 'warplane', 'killzones', 'invader', 'weapon', 'landmines', 'lawlessnesses', 'target', 'destructeds', 'munition', 'hazardouses', 'existentials', 'guerrillas', 'narcotic', 'atomic

In [17]:
# now these are displayed for visual inspection
all_categorised_securised_word_and_terms= [
    ("Threat Words", threat_seed_lexicon_words_signal_categories),
    ("Threat Terms", threat_seed_lexicon_terms_signal_categories),
    ("Urgency Words", urgency_seed_lexicon_words_signal_categories),
    ("Urgency Terms", urgency_seed_lexicon_terms_signal_categories),
    ("Justification Words", justification_of_extr_power_seed_lexicon_words_signal_categories),
    ("Justification Terms", justification_of_extr_power_seed_lexicon_terms_signal_categories)
]
pd.set_option('display.max_colwidth', None) # (looks better in dynamic sizing)
pd.set_option('display.max_rows', None)

# here I define the custom order for signal type
tier_order_to_display= ['core', 'secondary', 'contextual', 'n/a']

for name_of_lexicon, data_dict in all_categorised_securised_word_and_terms:
    lex_df= pd.DataFrame(list(data_dict.items()), columns=[name_of_lexicon.split()[0], 'Signal Type'])
    lex_df['Signal Type']= pd.Categorical(lex_df['Signal Type'], categories=tier_order_to_display, ordered=True)
    lex_df= lex_df.sort_values('Signal Type')
    print(f"\n Now I've categorised {name_of_lexicon}:")
    display(lex_df)


 Now I've categorised Threat Words:


,Threat,Signal Type
250,extinction,core
306,attacks,core
173,survival,core
344,wound,core
238,extremist,core
624,deaths,core
29,wars,core
594,monsters,core
388,extremism,core
226,drownings,core



 Now I've categorised Threat Terms:


,Threat,Signal Type
20,cogs of wars,core
35,war engine,core
45,war machine,core
33,existential threat,core
32,cogs of war,core
31,existential threats,core
29,serious threat,core
25,war machines,core
18,engine of war,core
16,war engines,core



 Now I've categorised Urgency Words:


...

,Urgency,Signal Type
44,urgent priorities,core
61,point of no return,core
2,act now,core
20,supreme priority,core
19,too late,core
7,supreme priorities,core
14,urgent priority,core
41,life threatening,core
28,mustn't delay,secondary
17,cannot delay,secondary



 Now I've categorised Justification Words:


,Justification,Signal Type
118,sieges,core
212,curfew,core
54,curfews,core
112,counterattacks,core
232,siege,core
145,counterattack,core
240,sovereign,secondary
239,prerogatives,secondary
106,decree,secondary
166,sovereigns,secondary



 Now I've categorised Justification Terms:


,Justification,Signal Type
45,state of emergency,core
28,state of emergencies,core
66,restore order,core
12,counter attack,core
62,martial laws,core
10,counter attacks,core
51,extraordinary means,core
69,extraordinary measures,core
41,fight back,core
74,martial law,core


## 2. Expansion using a lexicon from the literature
Now I expand my lexicon using Baele and Sterck’s Security Lexicon (Baele & Sterk, 2014), which is a specialised dictionary of 222 words they claim unambiguously belongs to the "hard" security lexical field. The made it to measure the linguistic intensity of securitising moves, so it fits well here. This primarily supports Threat Construction ("war," "enemy"), but also support the Justification of Extraordinary Measures one a bit too (e.g. "measures," "defend").

If not already identified in securitisation literature, I use these words only be used as secondary or contextual signals unless they stand out as excpetionally relevant to securitisation.

It uses stemming, so I'm seperating the lexicon into 2 list to deal with the stemmed variants seperately. I don't want to assume each stemmed variant is semantically equal in intonation (e.g. "remove" carries different weight to "removal"), so a degree of manual oversight is needed on my end to ensure all word variants are given the right category.

The words here are *generic* "hard security" ones (per Baele & Rousseau's wording) i.e. they directly evoke a security threat regardless of context.

In [18]:
non_stemmed_SL = {"abatement", "abm", "afghanistan", "aggression", "airland", "airpower", "alliance", "alert", "alqaeda", "ammunition", "amphibious", "anarchy",
           "antipersonnel", "apt", "arab", "assault", "atomic", "authority", "baader", "balance", "ballistic", "battle", "bin", "biological", "bullet",
           "ceasefire", "chechnya", "chemical", "cluster", "colonial", "compromise", "concession", "counterterrorism", "coup", "crisis", "critical",
           "cybersphere", "cyberwar", "détente", "dictator", "dilemma", "disaster", "disease", "disobedience", "dispute", "division", "drugs", "embargo",
           "emergency", "enemy", "escalation", "evil", "existential", "expeditionary", "faction", "failed", "fear", "firearms", "fln", "freedom", "friction",
           "gang", "gaza", "genocide", "globalization", "guerrilla", "guevara", "gulf", "hamas", "hard", "hezbollah", "hiroshima", "homeland", "hostage",
           "hussein", "identity", "illicit", "incumbent", "inhuman", "intelligence", "interdiction", "interpol", "intervention", "ira", "iran", "iraq",
           "landmine", "lebanon", "libya", "malacca", "missile", "muslims", "nagasaki", "narcotics", "nato", "nazi", "netwar", "nonproliferation", "nuclear",
           "offensive", "oil", "osce", "pakistan", "palestine", "partisan", "police", "poverty", "power", "rescue", "resolution", "resources", "response",
           "retaliation", "revenge", "revolution", "rogue", "safe", "salt", "sarin", "seapower", "shock", "somalia", "state", "strike", "suicide",
           "superpower", "surge", "surveillance", "syria", "target", "transnational", "uprising", "values", "vietnam", "water", "wmd"}

stemmed_SL = {"arm", "assassi", "attack", "bipolar", "blitzkrieg", "block", "bomb", "capabilit", "capitulat", "casualt", "catastroph", "clos", "coerc", "collaps",
       "combat", "compel", "conciliat", "conflict", "contain", "control", "cooperat", "counteract", "counterinsurgen", "countervail", "crim", "damag",
       "danger", "defen", "demilitaris", "demobilis", "destab", "destruct", "deter", "disarm", "disintegrat", "domin", "explos", "fight", "forbid",
       "force", "fundamentalis", "hazard", "hegemon", "illegal", "imminen", "infiltrat", "insecur", "insurgen", "interdependen", "invade", "israel",
       "kill", "kurd", "law", "liberat", "militar", "munition", "nationalis", "overwhelm", "preempt", "prevent", "proliferat", "protect", "puniti",
       "racis", "radical", "resist", "rival", "salw", "sanction", "scarc", "secur", "separatis", "shortage", "sovereign", "stabilis", "strateg", "tactic",
       "terror", "threat", "traffic", "violen", "war", "weapon", "wound"}

non_stemmed_SL.remove("hard") # (I thought this was too generic & will get too many false positives in manifesto context)

In [22]:
# All words already present in the lexicons can just be dropped
# This has got to done before assigning signal type to though, to avoid variation in tier ranking
lexicon_dicts= [threat_seed_lexicon_words_signal_categories, threat_seed_lexicon_terms_signal_categories, urgency_seed_lexicon_words_signal_categories, urgency_seed_lexicon_terms_signal_categories, justification_of_extr_power_seed_lexicon_words_signal_categories, justification_of_extr_power_seed_lexicon_terms_signal_categories]
def drop_dupes_from_lexicon_tier(set_to_filter_out):
  original_lex_set= set_to_filter_out.copy()
  for lex_dict in lexicon_dicts:
    redundancy_check= set(lex_dict.keys())
    set_to_filter_out= set_to_filter_out - redundancy_check
  print(f"I droppped {original_lex_set - set_to_filter_out} words.")
  print("these are the ones remaining:")
  return set_to_filter_out

# Get shot of the non-stemmed dupes
drop_dupes_from_lexicon_tier(non_stemmed_SL)

I droppped {'suicide', 'guerrilla', 'gulf', 'aggression', 'strike', 'nazi', 'superpower', 'surveillance', 'alqaeda', 'hamas', 'inhuman', 'nonproliferation', 'airpower', 'bullet', 'netwar', 'détente', 'drugs', 'osce', 'faction', 'cyberwar', 'disobedience', 'gang', 'shock', 'ballistic', 'abatement', 'biological', 'nagasaki', 'embargo', 'emergency', 'firearms', 'fln', 'assault', 'compromise', 'existential', 'dictator', 'nuclear', 'syria', 'muslims', 'rogue', 'critical', 'ira', 'uprising', 'division', 'partisan', 'nato', 'seapower', 'battle', 'hussein', 'coup', 'rescue', 'arab', 'concession', 'target', 'abm', 'chemical', 'enemy', 'antipersonnel', 'libya', 'evil', 'cybersphere', 'narcotics', 'sarin', 'chechnya', 'somalia', 'landmine', 'intervention', 'retaliation', 'hostage', 'gaza', 'iraq', 'interdiction', 'disaster', 'hezbollah', 'revenge', 'lebanon', 'iran', 'afghanistan', 'authority', 'interpol', 'escalation', 'cluster', 'crisis', 'genocide', 'power', 'palestine', 'atomic', 'disease', '

{'airland',
 'alert',
 'alliance',
 'ammunition',
 'amphibious',
 'apt',
 'baader',
 'balance',
 'bin',
 'dilemma',
 'dispute',
 'expeditionary',
 'failed',
 'fear',
 'freedom',
 'friction',
 'globalization',
 'identity',
 'incumbent',
 'intelligence',
 'malacca',
 'offensive',
 'oil',
 'police',
 'poverty',
 'resolution',
 'resources',
 'response',
 'revolution',
 'safe',
 'salt',
 'state',
 'surge',
 'transnational',
 'values',
 'water'}

In [23]:
# A list of commonly used terms are used here to produce variants of the stems provided by the SL
big_list_of_english_words= set(wordfreq.top_n_list("en", n=100000))

# I have this helper to search for matches given a root word
def expand_the_root_word(stem, vocab):
    return sorted({word for word in vocab if word.startswith(stem) and len(word) > len(stem)})

# I call the helper here to find all matches (which need manual vetting as I set n as quite high above, so some false +ve's are inevitable)
expanded_SL_lexicon_terms= {
    stem: expand_the_root_word(stem, big_list_of_english_words)
    for stem in stemmed_SL
}


# Now I create the data structure to hold the lexicon creation up to this point.
lexicon_rows= []
for root_word, variants_of_it in expanded_SL_lexicon_terms.items():
    for variant_of_word in variants_of_it:
        lexicon_rows.append({
            "root": root_word, "variant": variant_of_word,
            # these other 3 cols are useful when analysing these words in excel
            "keep": "", "tier": "",  # core, secondary & contextual signals
            "notes": "" # (just for my own purpose really, if I want to make a note about that particular word)
        })
stem_df= pd.DataFrame(lexicon_rows)
stem_df

,root,variant,keep,tier,notes
0,hegemon,hegemonic,,,
1,hegemon,hegemony,,,
2,invade,invaded,,,
3,invade,invader,,,
4,invade,invaders,,,
5,invade,invades,,,
6,illegal,illegality,,,
7,illegal,illegally,,,
8,illegal,illegals,,,
9,counteract,counteracted,,,


### Grading lexicon values
At this point I have a list of possible candidates for securitised language. Some of the stem variants will need to be dropped though because they don't resemble securitised language.

The remaining words then need to be assigned categories. As a reminder, primary signals are activley & unambigiously securitising, secondary signals activley intensify/accelarates/legitimises securitisation, whilst contextual signals condition interpretation but can aid securitisation in the right context.

To rank these, a human-in-the-loop approach is needed that doesn't soley rely on human opinion. Thus I use a gradingdefinition to account multiple converging signals but use human judement being the final decider.

The core question to be answered is: "If this word appears alone, does it push the utterance closer to a securitising move?"

However, whilst I take a common evalulation approach for all dimensions, this varies across securitisation dimensions, acknowledging their variation.

In [24]:
# Firstly, the non-securitising stems need to be dropped. This list was decided from a manual examination of the produced stems.
non_securitising_language= {
    "dangerfield", "salwa", "detergent", "detergents", "determinants",
    "determinate", "determination", "determinations", "determinative",
    "determinate", "determined", "determinedly", "determiner", "determines",
    "determining", "determinism", "deterministic", "domina", "dominatrix",
    "domine", "domineering", "domingo", "domingue", "dominguez", "domini",
    "dominic", "dominic's", "dominica", "dominican", "dominicans", "dominick",
    "dominik", "dominika", "dominique", "domino's", "dominus", "militarily",
    "cooperatives", "preventer", "resistor", "resistors", "lawal", "lawd",
    "lawes", "lawford", "lawgiver", "lawler", "lawley", "lawlor", "lawman",
    "lawmen", "lawn", "lawndale", "lawnmower", "lawnmowers", "lawns",
    "lawrence", "lawrence's", "lawrenceburg", "lawrenceville", "lawrie",
    "lawry", "lawson", "lawson's", "lawton", "lawyer", "lawyer's",
    "lawyering", "lawyers", "defenceman", "defencemen", "defendant",
    "defendant's", "defendants", "defense's", "defenseless", "defenseman",
    "defensemen", "defensible", "defensive", "defensively", "defensiveness",
    "defensor", "killa", "killah", "killam", "killarney", "killeen",
    "killen", "killian", "killick", "killin", "killington", "killjoy",
    "killjoys", "killua", "killy", "waratah", "waratahs", "warble",
    "warbler", "warblers", "warbling", "warburg", "warburton", "warby",
    "warcraft", "ward", "ward's", "warde", "warded", "wardell", "warden",
    "warden's", "wardens", "warder", "warders", "warding", "wardlaw",
    "wardle", "wardrobe", "wardrobes", "wards", "ware", "wareham",
    "warehouse", "warehouses", "warehousing", "wares", "warfarin",
    "warframe", "warg", "wargaming", "warhammer", "warhawk", "warhol",
    "warhol's", "warhorse", "warily", "wariness", "waring", "wario",
    "wark", "warlike", "warlock", "warlocks", "warlord", "warlords",
    "warm", "warmed", "warmer", "warmers", "warmest", "warming",
    "warminster", "warmly", "warms", "warmth", "warmup", "warmups",
    "warn", "warne", "warned", "warner", "warner's", "warners", "warning",
    "warnings", "warnock", "warns", "warp", "warpaint", "warped",
    "warping", "warps", "warr", "warrant", "warranted", "warranties",
    "warranting", "warrantless", "warrants", "warranty", "warren",
    "warren's", "warrens", "warrensburg", "warrenton", "warri", "warrick",
    "warriner", "warring", "warringah", "warrington", "warrior",
    "warrior's", "warriors", "warrnambool", "wars", "warsaw", "warsaw's",
    "warship", "warships", "warsi", "warszawa", "wart", "warthog",
    "warthogs", "warton", "warts", "warty", "warwick", "warwickshire",
    "wary", "warzone", "control's", "controller's", "controllers",
    "rivaldo", "fightin", "criminologist", "criminologists", "criminology",
    "crimp", "crimped", "crimping", "crimson", "block's", "blockage",
    "blockages", "blockbuster", "blockbusters", "blockchain", "blockchains",
    "blocker", "blockers", "blockhead", "blockheads", "blockhouse",
    "blocky", "forceps", "close", "closed", "closely", "closeness",
    "closeout", "closer", "closers", "closes", "closest", "closet",
    "closeted", "closets", "closeup", "closeups", "closing", "closings",
    "clostridium", "closure", "closures", "arm's", "arma", "armaan",
    "armada", "armadalearmadillo", "armadillos", "armagh", "armagnac",
    "arman", "armand", "armando", "armani", "armas", "armature", "armband",
    "armbands", "armbar", "armbruster", "armchair", "armchairs", "armee",
    "armen", "armenia", "armenia's", "armenian", "armenians", "armful",
    "armidale", "armie", "armin", "arminian", "arminius", "armisen",
    "armistead", "armitage", "armless", "armoire", "armond", "armonk",
    "armor", "armored", "armorial", "armories", "armors", "armpit",
    "armpits", "armrest", "armrests", "armstead", "armstrong",
    "armstrong's", "bomba", "bombast", "bombastic", "bombay",
    "bomberman", "container", "containers", "airland", "alert",
    "ammunition", "amphibious", "armadillo", "armadale", "baader",
    "balance", "bin", "law", "laws", "lawmaker", "lawmakers", "lawmaking",
    "lawsuit", "malacca", "salt", "securitization", "strategist", "strategists",
    "water", "hard",
}

# There are also some stems that should remain in their root form, that are dropped using the above method; here they're manually specified and re-included
securitising_stems= {"salw", "deter", "militarisation", "militarised", "uncooperating", "uncooperative", "uncooperativley", "strategise", "strategising", "resist",
                      "combat", "law", "weaponised", "weaponise", "sanctions", "counteract", "war", "counterinsurgent", "forbid", "rival", "shortage", "preempt",
                      "kurd", "israel", "countervail", "conflict", "terror", "radicalise", "radicalised", "radicalising", "radical", "contain", "hazard",
                      "abate", "disobedient", "esculate", "intervene", 'retaliate'}

# spellcheck (as these were manually typed) before dropping, or didn't include the English spelling (needed as these are British manifestos)
words_to_keep= {"salwa", "salw", "militarisation", "militarised", "weaponised", "weaponise", "strategise", "strategising", "radicalise", "radicalised",
                 "radicalising", "uncooperating", "uncooperativley", "killy", "warringah", "warhol's", "warde"}
spell_check_lexicon_words.known(list(words_to_keep))
#validate_lexicon_entries(non_securitising_language, "Non-securitising language (TO DROP)", words_to_keep)
validate_lexicon_entries(securitising_stems, "Securitising stems (TO RE-INCLUDE)", words_to_keep)

# Now the entire collection of valid roots and stems can be compiled
all_possible_variants= set(stem_df['variant'])
valid_variants= all_possible_variants - non_securitising_language
all_valid_stem_variants= securitising_stems.union(valid_variants).union(non_stemmed_SL)

print("\n Validated SL terms:")
display(all_valid_stem_variants)


 now spell checking for Securitising stems (TO RE-INCLUDE)...
the misspelled words in Securitising stems (TO RE-INCLUDE) are: {'esculate': 'escalate'}
so the corrected ones in Securitising stems (TO RE-INCLUDE) were: {'intervene', 'radicalised', 'counteract', 'rival', 'militarisation', 'israel', 'preempt', 'countervail', 'law', 'uncooperative', 'militarised', 'strategising', 'war', 'retaliate', 'contain', 'combat', 'forbid', 'radical', 'deter', 'terror', 'escalate', 'counterinsurgent', 'radicalise', 'abate', 'shortage', 'uncooperativley', 'weaponised', 'strategise', 'uncooperating', 'kurd', 'hazard', 'radicalising', 'weaponise', 'disobedient', 'conflict', 'sanctions', 'resist', 'salw'}
Great, the Securitising stems (TO RE-INCLUDE) lexicon is now all validated.

 Validated SL terms:


{'abate',
 'abatement',
 'abm',
 'afghanistan',
 'aggression',
 'airland',
 'airpower',
 'alert',
 'alliance',
 'alqaeda',
 'ammunition',
 'amphibious',
 'anarchy',
 'antipersonnel',
 'apt',
 'arab',
 'armageddon',
 'armament',
 'armaments',
 'armed',
 'armes',
 'armies',
 'arming',
 'armistice',
 'armory',
 'armour',
 'armoured',
 'armoury',
 'arms',
 'army',
 "army's",
 'armys',
 'assassin',
 "assassin's",
 'assassinate',
 'assassinated',
 'assassinating',
 'assassination',
 'assassinations',
 'assassins',
 'assault',
 'atomic',
 "attack's",
 'attacked',
 'attacker',
 "attacker's",
 'attackers',
 'attacking',
 'attacks',
 'authority',
 'baader',
 'balance',
 'ballistic',
 'battle',
 'bin',
 'biological',
 'blockade',
 'blockaded',
 'blockades',
 'blockading',
 'blocked',
 'blocking',
 'blocks',
 "bomb's",
 'bombard',
 'bombarded',
 'bombardier',
 'bombarding',
 'bombardment',
 'bombardments',
 'bombards',
 'bombed',
 'bomber',
 "bomber's",
 'bombers',
 'bombing',
 'bombings',
 'bombs

In [25]:
# Now these words need to be split into the 3 dimensions (threat, urgency & just. extr power)

# First Threat, which was identified using the question: "Does this term, when used,
# characterise a referent object or situation as dangerous, harmful, destabilising or existentially risky?"
threat_SL = {
    'abm', 'afghanistan', 'aggression', 'airpower', 'alqaeda', 'antipersonnel',
    'apt', 'arab', 'armament', 'armaments', 'armed', 'armes', 'armies', 'arming',
    'armory', 'armour', 'armoured', 'armoury', 'arms', 'army', 'armys',
    'assassin', 'assassins', 'assassinate', 'assassinated', 'assassinating',
    'assassination', 'assassinations', 'assault', 'atomic', 'attacks', 'attacked',
    'attacker', 'attackers', 'attacking', 'authority', 'ballistic', 'battle',
    'biological', 'blockade', 'blockaded', 'blockades', 'blockading', 'blocked',
    'blocking', 'blocks', 'bombs', 'bombard', 'bombarded', 'bombardier',
    'bombarding', 'bombardment', 'bombardments', 'bombards', 'bombed', 'bomber',
    'bombers', 'bombing', 'bombings', 'bombshell', 'bombshells', 'bullet',
    'capabilities', 'capability', 'capitulate', 'capitulated', 'capitulating',
    'capitulation', 'casualties', 'casualty', 'catastrophe', 'catastrophes',
    'catastrophic', 'catastrophically', 'chechnya', 'chemical', 'cluster',
    'coerce', 'coerced', 'coercing', 'coercion', 'coercive', 'colonial',
    'combat', 'combatant', 'combatants', 'combated', 'combating', 'combative',
    'combats', 'combatting', 'conflict', 'conflicted', 'conflicting', 'conflicts',
    'controller', 'coup', 'crime', 'crimea', 'crimean', 'crimes', 'crimestoppers',
    'crimewatch', 'criminal', 'criminals', 'criminalisation', 'criminalise',
    'criminalised', 'criminality', 'criminalization', 'criminalize', 'criminalized',
    'criminalizes', 'criminalizing', 'criminally', 'crisis', 'cybersphere',
    'cyberwar', 'destabilise', 'destabilising', 'destabilization', 'destabilize',
    'destabilized', 'destabilizing', 'destructed', 'destructible', 'destructing',
    'destruction', 'destructions', 'destructive', 'destructively', 'destructiveness',
    'destructor', 'deteriorate', 'deteriorated', 'deteriorates', 'deteriorating',
    'deterioration', 'determinant', 'determine', 'dictator', 'dilemma', 'disaster',
    'disease', 'disintegrate', 'disintegrated', 'disintegrates', 'disintegrating',
    'disintegration', 'disobedient', 'disobedience', 'dispute', 'division',
    'dominance', 'dominant', 'dominantly', 'dominate', 'dominated', 'dominates',
    'dominating', 'domination', 'dominator', 'dominators', 'dominion', 'dominions',
    'domino', 'dominoes', 'dominos', 'drugs', 'enemy', 'evil', 'existential',
    'explosion', 'explosions', 'explosive', 'explosively', 'explosiveness',
    'explosives', 'faction', 'fighter', 'fighters', 'fighting', 'fights',
    'firearms', 'fln', 'forces', 'forceful', 'forcefully', 'friction',
    'fundamentalism', 'fundamentalist', 'fundamentalists', 'gang', 'gaza',
    'genocide', 'globalization', 'guerrilla', 'guevara', 'gulf', 'hamas',
    'hazard', 'hazards', 'hazardous', 'hegemonic', 'hegemony',
    'hezbollah', 'hiroshima', 'hussein', 'illegality', 'illegally', 'illegals',
    'illicit', 'infiltrate', 'infiltrated', 'infiltrates', 'infiltrating',
    'infiltration', 'infiltrator', 'infiltrators', 'inhuman', 'insecure',
    'insecurities', 'insecurity', 'insurgencies', 'insurgency', 'insurgent',
    'insurgents', 'interpol', 'invaded', 'invader', 'invaders', 'invades',
    'ira', 'iran', 'iraq', 'israel', 'israels', 'israeli', 'israelis',
    'israelite', 'israelites', 'killed', 'killer', 'killers', 'killing',
    'killings', 'kills', 'killswitch', 'killzone', 'kurd', 'kurdish',
    'kurdistan', 'kurds', 'landmine', 'lawbreakers', 'lawless', 'lawlessness',
    'lebanon', 'libya', 'militaries', 'militarisation', 'militarised',
    'militarism', 'militarist', 'militaristic', 'militarization', 'militarized',
    'military', 'militarys', 'missile', 'munitions', 'muslims', 'nagasaki',
    'narcotics', 'nationalisation', 'nationalise', 'nationalised', 'nationalism',
    'nationalist', 'nationalistic', 'nationalists', 'nato', 'nazi', 'netwar',
    'nuclear', 'offensive', 'osce', 'overwhelmed', 'overwhelming', 'overwhelms',
    'pakistan', 'palestine', 'partisan', 'police', 'power', 'proliferate',
    'proliferated', 'proliferating', 'proliferation', 'proliferative', 'racism',
    'racist', 'racists', 'radical', 'radicalisation', 'radicalise', 'radicalised',
    'radicalising', 'radicalism', 'radicalization', 'radicalize', 'radicalized',
    'radicalizing', 'radically', 'radicals', 'revolution', 'rival', 'rivals',
    'rivaled', 'rivaling', 'rivalled', 'rivalling', 'rivalries', 'rivalry',
    'rogue', 'salw', 'salwar', 'sarin', 'seapower', 'separatism', 'separatist',
    'separatists', 'shock', 'shortage', 'shortages', 'somalia', 'state',
    'strike', 'suicide', 'superpower', 'surge', 'surveillance', 'syria',
    'target', 'terror', 'terrorise', 'terrorised', 'terrorising', 'terrorism',
    'terrorist', 'terrorists', 'terroristic', 'terrorize', 'terrorized',
    'terrorizes', 'terrorizing', 'terrors', 'threaten', 'threatened',
    'threatening', 'threateningly', 'threatens', 'threats', 'trafficked',
    'trafficker', 'traffickers', 'trafficking', 'traffics', 'transnational',
    'uprising', 'vietnam', 'violence', 'violent', 'violently', 'war', 'wars',
    'warfare', 'warfield', 'warfighter', 'warfighting', 'wargame', 'wargames',
    'warhead', 'warheads', 'warmonger', 'warmongering', 'warmongers', 'warpath',
    'warplane', 'warplanes', 'wartime', 'weapons', 'weaponise', 'weaponised',
    'weaponize', 'weaponized', 'weaponry', 'wmd', 'wounded', 'wounding', 'wounds'
}

# Then urgency, for which I used the question: "Does this term compress time, increase necessity or reduce deliberative space?"
urgency_SL = {
    "armageddon", "collapse", "collapsed", "collapses", "collapsible", "collapsing",
    "critical", "damage", "damaged", "damages", "damaging", "dangerous",
    "dangerously", "dangers", "emergency", "esculate", "escalation",
    "imminent", "imminently", "intervene", "intervention", "overwhelmingly",
    "scarce", "scarcely", "scarcers", "scarcity"
}

# Justification of Extraordinary Power, which was identified with:
# "Does this term legitimise actions that exceed or suspend normal political, legal or administrative procedures?"
just_extr_power_SL = {
    'abate', 'abatement', 'alliance', 'anarchy', 'armistice', 'ceasefire',
    'compelled', 'compelling', 'compellingly', 'compels', 'compromise',
    'concession', 'conciliation', 'conciliatory', 'contain', 'contained', 'containment',
    'controllability', 'controllable', 'controlled', 'controlling', 'controls',
    'cooperate', 'cooperated', 'cooperates', 'cooperating', 'cooperation',
    'cooperative', 'cooperatively', 'counteract', 'counteracted', 'counteracting',
    'counteracts', 'counterinsurgency', 'counterinsurgent', 'counterterrorism',
    'countervail', 'countervailing', 'defence', 'defenceless', 'defences',
    'defend', 'defended', 'defender', 'defenders', 'defending', 'defends',
    'defense', 'defenses', 'deter', 'deterred', 'deterrence', 'deterrent',
    'deterrents', 'deterring', 'deters', 'disarmament', 'disarmed', 'disarming',
    'disarms', 'détente', 'embargo', 'expeditionary', 'failed', 'fear',
    'fightback', 'forbid', 'forbidden', 'forbidding', 'forbids',
    'forced', 'freedom', 'homeland', 'hostage', 'identity', 'incumbent',
    'intelligence', 'interdependence', 'interdependent', 'interdiction',
    'lawful', 'lawfully', 'lawfulness', 'liberate', 'liberated', 'liberates',
    'liberating', 'liberation', 'liberator', 'liberators', 'nonproliferation',
    'oil', 'poverty', 'preempt', 'preempted', 'preemption', 'preemptive',
    'preemptively', 'preventable', 'preventative', 'prevented', 'preventing',
    'prevention', 'preventive', 'prevents', 'protected', 'protecting',
    'protection', 'protectionism', 'protectionist', 'protections', 'protective',
    'protectively', 'protector', 'protectorate', 'protectorates', 'protectors',
    'protects', 'punitive', 'rescue', 'resist', 'resistance', 'resistances',
    'resistant', 'resisted', 'resisters', 'resisting', 'resistive', 'resistivity',
    'resists', 'resolution', 'resources', 'response', 'retaliation', 'revenge',
    'safe', 'sanctioned', 'sanctioning', 'sanctions', 'secure', 'secured',
    'securely', 'secures', 'securing', 'securities', 'security', 'securitys',
    'sovereigns', 'sovereignty', 'stabilisation', 'stabilise', 'stabilised',
    'stabiliser', 'stabilisers', 'stabilising', 'strategic', 'strategical',
    'strategically', 'strategies', 'strategise', 'strategising', 'strategize',
    'strategizing', 'strategy', 'tactical', 'tactically', 'tactician', 'tactics',
    'uncooperating', 'uncooperative', 'uncooperativley', 'values', 'retaliate'
}

# Note that although there are a lot of new words, many of these are variations of roots & spellings on the same word
print(f"Total words in threat_SL: {len(threat_SL)}")
print(f"Total words in urgency_SL: {len(urgency_SL)}")
print(f"Total words in just_extr_power_SL: {len(just_extr_power_SL)}")

Total words in threat_SL: 401
Total words in urgency_SL: 26
Total words in just_extr_power_SL: 173


In [26]:
# And now they need to be assigned categories using the same rules defined above for the seed lexicons
SL_threat_words_core_signals = {'genocide', 'assassin', 'assassins', 'assassinate', 'assassinated', 'assassinating', 'assassination', 'assassinations',
                                'assault', 'atomic', 'attacks', 'attacked', 'attacker', 'attackers', 'attacking', 'authority', 'ballistic', 'battle',
                                'bombs', 'bombard', 'bombarded', 'bombardier', 'bombarding', 'bombardment', 'bombardments', 'bombards', 'bombed', 'bomber',
                                'bombers', 'bombing', 'bombings', 'bombshell', 'bombshells', 'bullet', 'casualties', 'casualty', 'catastrophe', 'catastrophes',
                                'catastrophic', 'catastrophically', 'combat', 'combatant', 'combatants', 'combated', 'combating', 'combative', 'combats',
                                'combatting', 'conflict', 'conflicted', 'conflicting', 'conflicts', 'coup', 'cyberwar', 'fighter', 'fighters', 'fighting',
                                'fights', 'firearms', 'forces', 'infiltrate', 'infiltrated', 'infiltrates', 'infiltrating', 'infiltration', 'infiltrator',
                                'infiltrators', 'insurgencies', 'insurgency', 'insurgent', 'insurgents', 'invaded', 'invader', 'invaders', 'invades', 'killed',
                                'killer', 'killers', 'killing', 'killings', 'kills', 'killswitch', 'killzone', 'landmine', 'explosion', 'explosions', 'explosive',
                                'explosively', 'explosiveness', 'explosives', 'missile', 'netwar', 'nuclear', 'suicide', 'violence', 'violent', 'violently', 'war', 'wars',
                                'warfare', 'warfield', 'warfighter', 'warfighting', 'wargame', 'wargames', 'warheads', 'warmonger', 'warmongering', 'warmongers', 'warpath',
                                'warplane', 'warplanes', 'wartime', 'weapons', 'weaponise', 'weaponised', 'weaponize', 'weaponized', 'weaponry', 'warhead'}

SL_threat_words_secondary_signals = {'aggression', 'airpower', 'antipersonnel',
    'armament', 'armaments', 'armed', 'armes', 'armies', 'arming',
    'armory', 'armour', 'armoured', 'armoury', 'arms', 'army', 'armys',
    'biological', 'blockade', 'blockaded', 'blockades', 'blockading', 'blocked',
    'blocking', 'blocks',
    'capabilities', 'capability', 'capitulate', 'capitulated', 'capitulating',
    'capitulation', 'chemical', 'cluster',
    'coerce', 'coerced', 'coercing', 'coercion', 'coercive', 'colonial',
    'controller', 'crime', 'crimes', 'crimestoppers',
    'crimewatch', 'criminal', 'criminals', 'criminalisation', 'criminalise',
    'criminalised', 'criminality', 'criminalization', 'criminalize', 'criminalized',
    'criminalizes', 'criminalizing', 'criminally', 'crisis', 'cybersphere',
    'destabilise', 'destabilising', 'destabilization', 'destabilize',
    'destabilized', 'destabilizing', 'destructed', 'destructible', 'destructing',
    'destruction', 'destructions', 'destructive', 'destructively', 'destructiveness',
    'destructor', 'deteriorate', 'deteriorated', 'deteriorates', 'deteriorating',
    'deterioration', 'determinant', 'determine', 'dictator', 'dilemma', 'disaster',
    'disease', 'disintegrate', 'disintegrated', 'disintegrates', 'disintegrating',
    'disintegration', 'disobedient', 'disobedience', 'dispute', 'division',
    'dominance', 'dominant', 'dominantly', 'dominate', 'dominated', 'dominates',
    'dominating', 'domination', 'dominator', 'dominators', 'dominion', 'dominions',
    'domino', 'dominoes', 'dominos', 'drugs', 'enemy', 'evil', 'existential',
    'faction', 'forceful', 'forcefully', 'friction',
    'fundamentalism', 'fundamentalist', 'fundamentalists', 'gang', 'globalization',
    'hazard', 'hazards', 'hazardous', 'hegemonic', 'hegemony',
    'illegality', 'illegally', 'illegals',
    'illicit',  'inhuman', 'insecure',
    'insecurities', 'insecurity', 'interpol', 'lawbreakers', 'lawless', 'lawlessness',
    'militaries', 'militarisation', 'militarised',
    'militarism', 'militarist', 'militaristic', 'militarization', 'militarized',
    'military', 'militarys', 'munitions',
    'nationalisation', 'nationalise', 'nationalised', 'nationalism',
    'nationalist', 'nationalistic', 'nationalists',  'offensive', 'overwhelmed', 'overwhelming', 'overwhelms',
    'partisan', 'police', 'power', 'proliferate',
    'proliferated', 'proliferating', 'proliferation', 'proliferative', 'racism',
    'racist', 'racists', 'radical', 'radicalisation', 'radicalise', 'radicalised',
    'radicalising', 'radicalism', 'radicalization', 'radicalize', 'radicalized',
    'radicalizing', 'radically', 'radicals', 'revolution', 'rival', 'rivals',
    'rivaled', 'rivaling', 'rivalled', 'rivalling', 'rivalries', 'rivalry',
    'rogue',  'seapower', 'separatism', 'separatist',
    'separatists', 'shock', 'shortage', 'shortages', 'state',
    'strike', 'superpower', 'surge', 'surveillance', 'syria',
    'target', 'terror', 'terrorise', 'terrorised', 'terrorising', 'terrorism',
    'terrorist', 'terrorists', 'terroristic', 'terrorize', 'terrorized',
    'terrorizes', 'terrorizing', 'terrors', 'threaten', 'threatened',
    'threatening', 'threateningly', 'threatens', 'threats', 'trafficked',
    'trafficker', 'traffickers', 'trafficking', 'traffics', 'transnational',
    'uprising', 'wounded', 'wounding', 'wounds'}

SL_threat_words_contextual_signals = {'abm', 'afghanistan', 'alqaeda', 'apt', 'arab', 'crimea', 'crimean', 'fln', 'gaza', 'guerrilla', 'guevara', 'gulf',
    'hamas', 'hezbollah', 'hiroshima', 'hussein', 'ira', 'iran', 'iraq', 'israel', 'israels', 'israeli', 'israelis',
    'israelite', 'israelites', 'kurd', 'kurdish', 'kurdistan', 'kurds', 'lebanon', 'libya', 'muslims', 'nagasaki', 'narcotics',
    'nato', 'nazi', 'pakistan', 'palestine', 'salw', 'vietnam', 'osce', 'somalia', 'salwar', 'sarin', 'wmd', 'chechnya'}


SL_urgency_words_core_signals = {"armageddon", "collapse", "collapsed", "collapses", "collapsible", "collapsing", "imminent", "imminently"}
SL_urgency_words_secondary_signals = {"critical", "damage", "damaged", "damages", "damaging", "dangerous", "dangerously", "dangers", "emergency", "esculate",
    "escalation", "intervene", "intervention", "overwhelmingly", "scarce", "scarcely", "scarcers", "scarcity"}
SL_urgency_words_contextual_signals = {}


SL_justification_of_extr_power_words_core_signals = {'anarchy', 'counterinsurgency', 'counterinsurgent', 'counterterrorism', 'countervail', 'countervailing',
    'defences', 'counteract', 'counteracted', 'counteracting', 'counteracts', 'defence',
    'defenceless', 'defend', 'defended', 'defender', 'defenders', 'defending', 'defends', 'defense',
    'defenses', 'deter', 'deterred', 'deterrence', 'deterrent', 'deterrents', 'deterring', 'deters',
    'disarmament', 'disarmed', 'disarming', 'disarms', 'fightback', 'forbid', 'forbidden', 'forbidding', 'forbids',
    'forced', 'freedom', 'homeland', 'hostage', 'interdiction', 'liberate', 'liberated', 'liberates',
    'liberating', 'liberation', 'liberator', 'liberators', 'nonproliferation', 'punitive', 'rescue', 'resist',
    'resistance', 'resistances', 'resistant', 'resisted', 'resisters', 'resisting', 'resistive', 'resistivity',
    'resists', 'resolution', 'retaliation', 'retaliate', 'revenge', 'sanctioned', 'sanctioning', 'sanctions',
    'stabilisation', 'stabilise', 'stabilised', 'stabiliser', 'stabilisers', 'stabilising', 'tactical',
    'tactically', 'tactician', 'tactics'}
#SL_justification_of_extr_power_terms_core_signals = {'fight back'}

SL_justification_of_extr_power_words_secondary_signals = {'abate', 'abatement', 'alliance', 'compelled', 'compelling', 'compellingly', 'compels', 'compromise',
    'contain', 'contained', 'containment', 'controllability', 'controllable', 'controlled', 'controlling',
    'controls', 'cooperate', 'cooperated', 'cooperates', 'cooperating', 'cooperation', 'cooperative',
    'cooperatively', 'fear', 'incumbent', 'interdependence', 'interdependent', 'preempt', 'preempted', 'preemption', 'preemptive',
    'preemptively', 'preventable', 'preventative', 'prevented', 'preventing', 'prevention', 'preventive',
    'prevents', 'protected', 'protecting', 'protection', 'protectionism', 'protectionist', 'protections', 'protective',
    'protectively', 'protector', 'protectorate', 'protectorates', 'protectors', 'protects', 'response',
    'safe', 'secure', 'secured', 'securely', 'secures', 'securing', 'sovereigns', 'sovereignty',
    'strategic', 'strategical', 'strategically', 'strategies', 'strategise', 'strategising', 'strategize',
    'strategizing', 'strategy', 'values', 'uncooperative', 'uncooperating', 'uncooperativley'}

SL_justification_of_extr_power_words_contextual_signals = {'armistice', 'ceasefire', 'concession', 'conciliation', 'conciliatory', 'détente', 'embargo',
    'expeditionary', 'failed', 'identity', 'intelligence', 'lawful', 'lawfully', 'lawfulness', 'oil',
    'poverty', 'resources', 'securities', 'security', 'securitys'}

In [27]:
# Then these are to be combined assigned signals and combined with the seed word lexicons

# 1st I categorise SL (seed lexicon) Threat Words
SL_threat_seed_lexicon_words_signal_categories = categorise_and_report_lexicon(
    threat_SL, SL_threat_words_core_signals, SL_threat_words_secondary_signals, SL_threat_words_contextual_signals, "SL threat words")

# then urgency
SL_urgency_seed_lexicon_words_signal_categories = categorise_and_report_lexicon(
    urgency_SL, SL_urgency_words_core_signals, SL_urgency_words_secondary_signals, SL_threat_words_contextual_signals, "SL urgency words")

# finally justification of extraordinary power
SL_justification_of_extr_power_seed_lexicon_words_signal_categories = categorise_and_report_lexicon(
    just_extr_power_SL, SL_justification_of_extr_power_words_core_signals, SL_justification_of_extr_power_words_secondary_signals, SL_justification_of_extr_power_words_contextual_signals, "SL justification words")

All good in SL threat words, I've assigned each term a signal category
All good in SL urgency words, I've assigned each term a signal category
All good in SL justification words, I've assigned each term a signal category


In [28]:
# now I combine SL signal categories with existing seed lexicon signal categories

# Threat words
threat_seed_lexicon_words_signal_categories.update(SL_threat_seed_lexicon_words_signal_categories)

# Urgency words
urgency_seed_lexicon_words_signal_categories.update(SL_urgency_seed_lexicon_words_signal_categories)

# Justification of Extraordinary Power words
justification_of_extr_power_seed_lexicon_words_signal_categories.update(SL_justification_of_extr_power_seed_lexicon_words_signal_categories)

print("Combined signal categories successfully!")
print(f"The updated Threat words count is: {len(threat_seed_lexicon_words_signal_categories)}")
print(f"& Urgency word count is at: {len(urgency_seed_lexicon_words_signal_categories)}")
print(f"& Justification at: {len(justification_of_extr_power_seed_lexicon_words_signal_categories)}")

Combined signal categories successfully!
The updated Threat words count is: 707
& Urgency word count is at: 79
& Justification at: 331


# 3. Exporting lexicons ready for SBERT expansion
At this stage, the lexicon is substaintial and highly targeted. I've gone from seed dictionary -> expanded with known securitisation literature. Now the final lexicon is to be exported with the aim of now searching for semantically similar words in the same neighbourhoods as the current lexicon words, which will be done with SBERT.

Expansion will prioritise human-in-the-loop oversight to ensure the lexcion dimensions are expanded precisely and minimally. Importantly, embedding-based expansion will not alter the theory-derived lexicon above, which is now frozen.

In [ ]:
# Now these are exported to an excel for SBERT expansion (in another collab file)
threat_data = {"Term": list(threat_seed_lexicon_words_signal_categories.keys()) + list(threat_seed_lexicon_terms_signal_categories.keys()),
               "Signal Type": list(threat_seed_lexicon_words_signal_categories.values()) + list(threat_seed_lexicon_terms_signal_categories.values())}
df_threat= pd.DataFrame(threat_data)

urgency_data= {"Term": list(urgency_seed_lexicon_words_signal_categories.keys()) + list(urgency_seed_lexicon_terms_signal_categories.keys()),
                "Signal Type": list(urgency_seed_lexicon_words_signal_categories.values()) + list(urgency_seed_lexicon_terms_signal_categories.values())}
df_urgency= pd.DataFrame(urgency_data)

justification_data= {"Term": list(justification_of_extr_power_seed_lexicon_words_signal_categories.keys()) + list(justification_of_extr_power_seed_lexicon_terms_signal_categories.keys()),
                      "Signal Type": list(justification_of_extr_power_seed_lexicon_words_signal_categories.values()) + list(justification_of_extr_power_seed_lexicon_terms_signal_categories.values())}
df_justification= pd.DataFrame(justification_data)

# here I'm converting the 'Signal Type' col to a categorical type with custom order and sorting the dfs
df_threat['Signal Type']= pd.Categorical(df_threat['Signal Type'], categories=tier_order_to_display, ordered=True)
df_threat= df_threat.sort_values('Signal Type').reset_index(drop=True)

df_urgency['Signal Type']= pd.Categorical(df_urgency['Signal Type'], categories=tier_order_to_display, ordered=True)
df_urgency = df_urgency.sort_values('Signal Type').reset_index(drop=True)

df_justification['Signal Type']= pd.Categorical(df_justification['Signal Type'], categories=tier_order_to_display, ordered=True)
df_justification= df_justification.sort_values('Signal Type').reset_index(drop=True)

created_lexicon_output_filename= "securitisation_lexicon.xlsx"

with pd.ExcelWriter(created_lexicon_output_filename, engine='xlsxwriter') as writer:
    df_threat.to_excel(writer, sheet_name='Threat Lexicon', index=False)
    df_urgency.to_excel(writer, sheet_name='Urgency Lexicon', index=False)
    df_justification.to_excel(writer, sheet_name='Justification Lexicon', index=False)

print(f"Lexicons exported to {created_lexicon_output_filename} successfully!")

## References I used
Buzan, Barry, Ole Wæver, and Jaap de Wilde. *Security: A New Framework for Analysis*. Boulder, CO: Lynne Rienner Publishers, 1998.

Baele, S. J., & Rousseau, E. (2022). At war or saving lives? On the securitizing semantic repertoires of Covid-19. International Relations, 37(2), 201–227. https://doi.org/10.1177/00471178221122957

‌Otukoya, T. A. (2024). The securitization theory. International Journal of Science and Research Archive, 11(1), 1747–1755. https://doi.org/10.30574/ijsra.2024.11.1.0225

Ahmad, Y., & Bouwman, B. (2025). MA Thesis Securitisation and Nubian Erasure in Postcolonial Egypt. Retrieved January 16, 2026, from https://studenttheses.uu.nl/bitstream/handle/20.500.12932/50484/Securitisation%20and%20Nubian%20Erasure%20in%20Postcolonial%20Egypt%20-%20Youssef%20Ahmad.pdf?sequence=1&isAllowed=y

State of Exception. (2003). http://pdf-objects.com/files/US-English-PDF-Object.pdf

Flohr, M. (2025, March 31). Key Concept: Securitization (Copenhagen School). Critical Legal Thinking. https://criticallegalthinking.com/2025/03/31/key-concept-securitization-copenhagen-school/

‌Baele, S. J., & Sterck, O. C. (2014). Diagnosing the Securitisation of Immigration at the EU Level: A New Method for Stronger Empirical Claims. Political Studies, 63(5), 1120–1139. https://doi.org/10.1111/1467-9248.12147

Duan, Z., Shao, A., Hu, Y., Lee, H., Liao, X., Suh, Y. J., Kim, J., Yang, K.-C., Chen, K., & Yang, S. (2025). Constructing Vec-tionaries to Extract Message Features from Texts: A Case Study of Moral Content. Political Analysis, 33(4), 425–445. https://doi.org/10.1017/pan.2025.6

S. Stetter, E. Herschinger, T. Teichler, and M. Albert, “Conflicts about water: Securitizations in a global context,” Jstor.org, 2026, doi: https://doi.org/10.2307/45084673.
‌